# Group 2 — Data Checks Before Writing the Report
**Targets: systolic/diastolic blood pressure (`upper_ap`, `lower_ap`) and oxygen saturation (`saturation`)**

This notebook confirms, on the real data, every pattern or anomaly discussed so far, so the weekly report
states findings backed by code rather than by a five-subject sample or a screenshot. Each section:
states **what is being checked**, **why it matters**, and prints a **pass/fail-style verdict**.

**Before running:** set `DATA_DIR` below to the folder that contains `db.csv` and the `ecg/`, `ppg/`,
`ppg_sync/`, `video/`, `meta/` subfolders (e.g. the folder Git Bash cloned into, such as
`C:/Users/<you>/mcd_rppg`).


In [2]:
import os
import json
import datetime
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from scipy.signal import find_peaks
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False
    print("scipy not found — install with: pip install scipy --break-system-packages")

# ---- CONFIG: change this to your local dataset folder ----
DATA_DIR = Path(r"C:\Users\User\mcd_rppg")  # e.g. Path("C:/Users/<you>/mcd_rppg")
DB_CSV = DATA_DIR / "db.csv"

assert DB_CSV.exists(), f"Could not find {DB_CSV} — update DATA_DIR above."
print("Using data dir:", DATA_DIR.resolve())


Using data dir: C:\Users\User\mcd_rppg


## 1. Load `db.csv` and confirm basic structure
Checks: row/column counts, unique subjects, cameras, views, steps — the structural facts your
report states as given.


In [3]:
df = pd.read_csv(DB_CSV)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print()
print("Unique patients:", df.patient_id.nunique())
print("Unique cameras:", df.camera.unique().tolist())
print("Unique views:", df.view.unique().tolist())
print("Unique steps:", df.step.unique().tolist())

expected_rows = 3600
expected_subjects = 600
print()
print("CHECK rows == 3600:", "PASS" if len(df) == expected_rows else f"FAIL (got {len(df)})")
print("CHECK subjects == 600:", "PASS" if df.patient_id.nunique() == expected_subjects else f"FAIL (got {df.patient_id.nunique()})")


Shape: (3600, 25)
Columns: ['patient_id', 'weight', 'height', 'bmi', 'age', 'sex', 'upper_ap', 'lower_ap', 'saturation', 'temperature', 'hemoglobin', 'glycated_hemoglobin', 'cholesterol', 'respiratory', 'rigidity', 'pulse', 'stress', 'step', 'camera', 'view', 'ecg', 'ppg', 'video', 'meta', 'ppg_sync']

Unique patients: 600
Unique cameras: ['FullHDwebcam', 'USBVideo', 'IriunWebcam']
Unique views: ['front', 'left', 'right']
Unique steps: ['after', 'before']

CHECK rows == 3600: PASS
CHECK subjects == 600: PASS


## 2. Confirm glucose is genuinely absent
Both your own scan and Jos's report found no glucose column. Confirm this programmatically
(not just by eye) and list all column names so nothing is missed.


In [4]:
glucose_like = [c for c in df.columns if "gluc" in c.lower()]
print("Columns containing 'gluc':", glucose_like)
print("CHECK glucose column present:", "FAIL — glucose found" if glucose_like else "PASS — confirmed absent")
print()
print("Full column list for reference:")
for c in df.columns:
    print(" -", c)


Columns containing 'gluc': []
CHECK glucose column present: PASS — confirmed absent

Full column list for reference:
 - patient_id
 - weight
 - height
 - bmi
 - age
 - sex
 - upper_ap
 - lower_ap
 - saturation
 - temperature
 - hemoglobin
 - glycated_hemoglobin
 - cholesterol
 - respiratory
 - rigidity
 - pulse
 - stress
 - step
 - camera
 - view
 - ecg
 - ppg
 - video
 - meta
 - ppg_sync


## 3. Descriptive statistics for your two targets
Full-dataset stats for `upper_ap` (systolic), `lower_ap` (diastolic), and `saturation`,
confirming the ranges and distribution shape reported earlier.


In [5]:
for col in ["upper_ap", "lower_ap", "saturation"]:
    print(f"--- {col} ---")
    print(df[col].describe())
    print()


--- upper_ap ---
count    3600.000000
mean      122.455000
std        17.429784
min        80.000000
25%       109.000000
50%       120.000000
75%       132.000000
max       202.000000
Name: upper_ap, dtype: float64

--- lower_ap ---
count    3600.00000
mean       73.79250
std         9.25166
min        50.00000
25%        67.75000
50%        73.00000
75%        80.00000
max       108.00000
Name: lower_ap, dtype: float64

--- saturation ---
count    3600.000000
mean       98.008333
std         1.291147
min        86.000000
25%        98.000000
50%        98.000000
75%        99.000000
max        99.000000
Name: saturation, dtype: float64



In [6]:
print("Saturation value counts (sorted):")
print(df["saturation"].value_counts().sort_index())
print()
top_values = df["saturation"].value_counts().sort_index(ascending=False).head(3)
pct_top = df["saturation"].isin([98.0, 99.0]).mean() * 100
print(f"CHECK: % of rows with saturation in (98, 99): {pct_top:.1f}%")
print("This is the 'narrow range' finding — flag in the report if > 50%.")


Saturation value counts (sorted):
saturation
86.0       3
88.0       3
91.0       6
92.0       6
93.0       9
94.0      36
95.0     138
96.0     168
97.0     465
98.0    1188
99.0    1578
Name: count, dtype: int64

CHECK: % of rows with saturation in (98, 99): 76.8%
This is the 'narrow range' finding — flag in the report if > 50%.


## 4. Before/after exercise comparison — full 600 subjects
Confirms, at full scale (not the earlier 5-subject sample), how much each target changes
between the 'before' and 'after' exercise states.


In [7]:
sub = df.drop_duplicates(["patient_id", "step"])[
    ["patient_id", "step", "upper_ap", "lower_ap", "saturation", "pulse"]
]
pivot = sub.pivot(index="patient_id", columns="step",
                   values=["upper_ap", "lower_ap", "saturation", "pulse"])
pivot.columns = ["_".join(c) for c in pivot.columns]

pivot["upper_diff"] = pivot["upper_ap_after"] - pivot["upper_ap_before"]
pivot["lower_diff"] = pivot["lower_ap_after"] - pivot["lower_ap_before"]
pivot["sat_diff"] = pivot["saturation_after"] - pivot["saturation_before"]

print("Before -> after change, across all", len(pivot), "subjects:")
print(pivot[["upper_diff", "lower_diff", "sat_diff"]].describe())
print()

# CHECK: did every subject actually have both a before and after recording?
missing_before = pivot["upper_ap_before"].isna().sum()
missing_after = pivot["upper_ap_after"].isna().sum()
print(f"CHECK subjects missing a 'before' row: {missing_before}")
print(f"CHECK subjects missing an 'after' row: {missing_after}")
print("(Non-zero counts here mean some subjects only have one state — note this if found.)")


Before -> after change, across all 600 subjects:
       upper_diff  lower_diff    sat_diff
count  600.000000  600.000000  600.000000
mean    10.343333    1.715000   -0.236667
std     13.745922    7.952096    1.402593
min    -24.000000  -33.000000  -13.000000
25%      0.000000   -3.000000   -1.000000
50%      9.000000    1.000000    0.000000
75%     19.000000    6.000000    0.000000
max     63.000000   38.000000   11.000000

CHECK subjects missing a 'before' row: 0
CHECK subjects missing an 'after' row: 0
(Non-zero counts here mean some subjects only have one state — note this if found.)


## 5. Static vs dynamic targets — does label leakage risk hold at full scale?
Earlier (5-subject) finding: some targets (cholesterol, hemoglobin, stress, age, sex, BMI)
barely change before/after, so a model could succeed by recognising the *person* rather than
measuring physiology. Confirm this pattern holds across all subjects, and quantify it.


In [8]:
candidate_static_cols = [
    "cholesterol", "hemoglobin", "glycated_hemoglobin", "stress", "age", "bmi"
]
sub2 = df.drop_duplicates(["patient_id", "step"])
for col in candidate_static_cols:
    if col not in df.columns:
        print(f"(column not found: {col})")
        continue
    p = sub2.pivot(index="patient_id", columns="step", values=col)
    if "before" not in p.columns or "after" not in p.columns:
        continue
    diff = (p["after"] - p["before"]).dropna()
    pct_unchanged = (diff == 0).mean() * 100
    print(f"{col:22s}: {pct_unchanged:5.1f}% of subjects identical before/after "
          f"(mean |diff| = {diff.abs().mean():.3f})")
print()
print("Compare against your two targets (expected to vary much more):")
for col in ["upper_ap", "lower_ap", "saturation"]:
    p = sub2.pivot(index="patient_id", columns="step", values=col)
    diff = (p["after"] - p["before"]).dropna()
    pct_unchanged = (diff == 0).mean() * 100
    print(f"{col:22s}: {pct_unchanged:5.1f}% identical before/after "
          f"(mean |diff| = {diff.abs().mean():.3f})")


cholesterol           :  99.8% of subjects identical before/after (mean |diff| = 0.002)
hemoglobin            : 100.0% of subjects identical before/after (mean |diff| = 0.000)
glycated_hemoglobin   :  99.7% of subjects identical before/after (mean |diff| = 0.000)
stress                :  99.3% of subjects identical before/after (mean |diff| = 0.010)
age                   : 100.0% of subjects identical before/after (mean |diff| = 0.000)
bmi                   :  99.8% of subjects identical before/after (mean |diff| = 0.002)

Compare against your two targets (expected to vary much more):
upper_ap              :   4.7% identical before/after (mean |diff| = 13.120)
lower_ap              :   6.3% identical before/after (mean |diff| = 6.002)
saturation            :  47.8% identical before/after (mean |diff| = 0.790)


## 6. Low-saturation subjects — generate the list fresh
Rebuilds the list of lowest-saturation sessions directly from `db.csv`, so it can be
regenerated if the data changes, rather than relying on a copy-pasted table.


In [9]:
low_sat = (
    df.drop_duplicates(["patient_id", "step"])
    .sort_values("saturation")
    .head(10)[["patient_id", "step", "saturation", "upper_ap", "lower_ap", "ecg", "ppg"]]
)
print(low_sat.to_string(index=False))


 patient_id   step  saturation  upper_ap  lower_ap                  ecg                ppg
       4504  after        86.0     102.0      66.0  ecg/4504_after.json  ppg/4504_after.PW
       3658 before        88.0      96.0      68.0 ecg/3658_before.json ppg/3658_before.PW
       7736  after        91.0     148.0      78.0  ecg/7736_after.json  ppg/7736_after.PW
       7137  after        91.0     107.0      62.0  ecg/7137_after.json  ppg/7137_after.PW
       6193 before        92.0     174.0      99.0 ecg/6193_before.json ppg/6193_before.PW
       5138  after        92.0     145.0      55.0  ecg/5138_after.json  ppg/5138_after.PW
       2916  after        93.0     136.0      88.0  ecg/2916_after.json  ppg/2916_after.PW
       6397  after        93.0     147.0      62.0  ecg/6397_after.json  ppg/6397_after.PW
       3529  after        93.0     114.0      78.0  ecg/3529_after.json  ppg/3529_after.PW
       5292 before        94.0     145.0      75.0 ecg/5292_before.json ppg/5292_before.PW

## 7. File readers for ECG (`.json`) and PPG (`.PW`)
Reusable functions. Run the signal-quality and heart-rate checks below on whatever subjects
you have downloaded locally — by default, the low-saturation subjects from Section 6.


In [10]:
def load_ecg(path):
    '''Return (frequency, {lead_title: np.array(values)}) from an ECG json file.'''
    d = json.loads(Path(path).read_text())
    leads = {item["title"]: np.array(item["values"]) for item in d["data"]}
    return d["frequency"], leads


def load_pw(path):
    '''Return (values, timestamps) from a .PW PPG file.'''
    vals, times = [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            vals.append(float(parts[0]))
            times.append(parts[1])
    return np.array(vals), times


def ppg_sampling_rate(times):
    fmt = "%Y-%m-%d %H:%M:%S.%f"
    t0 = datetime.datetime.strptime(times[0], fmt)
    t1 = datetime.datetime.strptime(times[-1], fmt)
    duration = (t1 - t0).total_seconds()
    fs = len(times) / duration if duration > 0 else None
    return fs, duration


def estimate_hr(signal, fs, min_bpm=40, max_bpm=180):
    '''Simple peak-based heart rate estimate. Returns (hr_bpm, n_peaks).'''
    if not HAVE_SCIPY or fs is None:
        return None, 0
    min_dist = int(fs * 60 / max_bpm)
    peaks, _ = find_peaks(signal, distance=max(min_dist, 1),
                           prominence=(signal.max() - signal.min()) * 0.2)
    if len(peaks) < 2:
        return None, len(peaks)
    ibi = np.diff(peaks) / fs
    hr = 60 / np.mean(ibi)
    if not (min_bpm <= hr <= max_bpm):
        return None, len(peaks)
    return hr, len(peaks)


## 8. ECG lead-length anomaly — confirm across many subjects
One session already showed lead I at 14,900 samples vs 15,000 for every other lead
(matching a finding in a teammate's report). Scan every ECG file you have locally to see how
widespread this is, rather than relying on a single confirmed case.


In [11]:
ecg_files = sorted((DATA_DIR / "ecg").glob("*.json")) if (DATA_DIR / "ecg").exists() else []
print(f"Found {len(ecg_files)} ECG files under {DATA_DIR / 'ecg'}")

lead_I_short_count = 0
checked = 0
mismatch_examples = []

for path in ecg_files:
    try:
        freq, leads = load_ecg(path)
    except Exception as e:
        print(f"  could not read {path.name}: {e}")
        continue
    checked += 1
    lengths = {title: len(vals) for title, vals in leads.items()}
    if len(set(lengths.values())) > 1:
        lead_I_short_count += 1
        if len(mismatch_examples) < 5:
            mismatch_examples.append((path.name, lengths))

print()
print(f"CHECK: files with mismatched lead lengths: {lead_I_short_count} / {checked}")
if mismatch_examples:
    print("Examples:")
    for name, lengths in mismatch_examples:
        print(" ", name, lengths)
print()
if checked == 0:
    print("No local ECG files found — download a batch into DATA_DIR/ecg/ before running this cell.")


Found 1200 ECG files under C:\Users\User\mcd_rppg\ecg
  could not read 5214_before.json: Extra data: line 1 column 3130462 (char 3130461)
  could not read 5904_before.json: Extra data: line 1 column 3100106 (char 3100105)
  could not read 8001_before.json: Extra data: line 1 column 3106692 (char 3106691)

CHECK: files with mismatched lead lengths: 1197 / 1197
Examples:
  1020_after.json {'I': 14900, 'II': 15000, 'V1': 15000, 'V2': 15000, 'V3': 15000, 'V4': 15000, 'V5': 15000, 'V6': 15000, 'III': 15000, 'aVR': 15000, 'aVL': 15000, 'aVF': 15000}
  1020_before.json {'I': 14900, 'II': 15000, 'V1': 15000, 'V2': 15000, 'V3': 15000, 'V4': 15000, 'V5': 15000, 'V6': 15000, 'III': 15000, 'aVR': 15000, 'aVL': 15000, 'aVF': 15000}
  1024_after.json {'I': 14900, 'II': 15000, 'V1': 15000, 'V2': 15000, 'V3': 15000, 'V4': 15000, 'V5': 15000, 'V6': 15000, 'III': 15000, 'aVR': 15000, 'aVL': 15000, 'aVF': 15000}
  1024_before.json {'I': 14900, 'II': 15000, 'V1': 15000, 'V2': 15000, 'V3': 15000, 'V4': 150

## 9. Signal quality — flat-line detection (PPG and ECG)
A teammate's report found two subjects with perfectly flat ('dead') synchronised PPG.
Run the same check yourself on whatever PPG (`.PW`) and ECG files you have locally, so the
exclusion is something you've verified, not just taken on trust.


In [12]:
def is_flat(signal, tol=1e-9):
    return np.ptp(signal) <= tol  # peak-to-peak range ~ 0


pw_files = sorted((DATA_DIR / "ppg").glob("*.PW")) if (DATA_DIR / "ppg").exists() else []
print(f"Found {len(pw_files)} PPG (.PW) files under {DATA_DIR / 'ppg'}")

flat_ppg = []
for path in pw_files:
    try:
        vals, _ = load_pw(path)
    except Exception as e:
        print(f"  could not read {path.name}: {e}")
        continue
    if is_flat(vals):
        flat_ppg.append(path.name)

print(f"CHECK: flat-line PPG files found: {len(flat_ppg)}")
if flat_ppg:
    print("Flagged files:", flat_ppg)
print()

flat_ecg = []
for path in ecg_files:
    try:
        freq, leads = load_ecg(path)
    except Exception:
        continue
    for title, vals in leads.items():
        if is_flat(vals):
            flat_ecg.append((path.name, title))

print(f"CHECK: flat ECG leads found: {len(flat_ecg)}")
if flat_ecg:
    print("Flagged (file, lead):", flat_ecg)


Found 1200 PPG (.PW) files under C:\Users\User\mcd_rppg\ppg
CHECK: flat-line PPG files found: 0

CHECK: flat ECG leads found: 5744
Flagged (file, lead): [('1020_after.json', 'V2'), ('1020_after.json', 'V3'), ('1020_after.json', 'V4'), ('1020_after.json', 'V5'), ('1020_after.json', 'V6'), ('1024_after.json', 'V2'), ('1024_after.json', 'V3'), ('1024_after.json', 'V4'), ('1024_after.json', 'V5'), ('1024_after.json', 'V6'), ('1035_after.json', 'V2'), ('1035_after.json', 'V3'), ('1035_after.json', 'V4'), ('1035_after.json', 'V5'), ('1035_after.json', 'V6'), ('1035_before.json', 'V2'), ('1035_before.json', 'V3'), ('1035_before.json', 'V4'), ('1035_before.json', 'V5'), ('1035_before.json', 'V6'), ('1091_after.json', 'V2'), ('1091_after.json', 'V3'), ('1091_after.json', 'V4'), ('1091_after.json', 'V5'), ('1091_after.json', 'V6'), ('1091_before.json', 'V2'), ('1091_before.json', 'V3'), ('1091_before.json', 'V4'), ('1091_before.json', 'V5'), ('1091_before.json', 'V6'), ('1097_after.json', 'V2'),

## 10. Heart rate: PPG-derived vs ECG-derived, for the low-saturation subjects
Confirms whether the low-saturation sessions have normal, plausible heart rates (evidence
the readings aren't sensor glitches) and cross-checks PPG against ECG lead II.


In [13]:
rows = []
for _, r in low_sat.iterrows():
    stem = f"{r.patient_id}_{r.step}"
    pw_path = DATA_DIR / r.ppg  # path already stored in db.csv, e.g. "ppg/1020_after.PW"
    ecg_path = DATA_DIR / r.ecg

    ppg_hr = ecg_hr = None
    fs = dur = None
    if pw_path.exists():
        vals, times = load_pw(pw_path)
        fs, dur = ppg_sampling_rate(times)
        ppg_hr, _ = estimate_hr(vals, fs)
    if ecg_path.exists():
        freq, leads = load_ecg(ecg_path)
        lead2 = leads.get("II")
        if lead2 is not None:
            ecg_hr, _ = estimate_hr(lead2, freq)

    rows.append({
        "subject": stem, "saturation": r.saturation,
        "ppg_fs": fs, "duration_s": dur,
        "ppg_hr": ppg_hr, "ecg_hr": ecg_hr,
        "ppg_file_found": pw_path.exists(), "ecg_file_found": ecg_path.exists(),
    })

hr_table = pd.DataFrame(rows)
print(hr_table.to_string(index=False))
print()
print("CHECK: any HR outside a plausible 40-180 bpm range, or missing, suggests the file")
print("needs a closer look before you treat the low-saturation reading as 'real'.")


    subject  saturation     ppg_fs  duration_s    ppg_hr    ecg_hr  ppg_file_found  ecg_file_found
 4504_after        86.0 100.006385  180.488476 93.578437       NaN            True            True
3658_before        88.0 100.007553  180.716351 67.544806       NaN            True            True
 7736_after        91.0 100.009240  180.783296 93.331587 82.941664            True            True
 7137_after        91.0 100.006787  180.567745 69.935596 71.750718            True            True
6193_before        92.0 100.008852  180.624011 79.112476 83.339121            True            True
 5138_after        92.0 100.006932  180.527485 86.094972 79.182458            True            True
 2916_after        93.0 100.000349  180.449371 87.185237 72.423783            True            True
 6397_after        93.0 100.011367  180.609470 55.129228 59.847286            True            True
 3529_after        93.0 100.008654  180.514379 66.393004 66.982409            True            True
5292_befor

## 11. Subject-level split — leakage self-test
Confirms the earlier finding that the same `patient_id` always carries the same label within
a state. If true, a naive video-level (not subject-level) split would leak identity between
train and test. This cell proves the leakage risk exists, and includes a worked example split.


In [14]:
# Does every camera, for a given subject+state, share identical target values?
cols_to_check = ["upper_ap", "lower_ap", "saturation"]
leak_check = df.groupby(["patient_id", "step"])[cols_to_check].nunique()
non_unique = (leak_check > 1).any(axis=1).sum()
print(f"CHECK: subject+state groups where targets differ across cameras: {non_unique} "
      f"(expected: 0)")
print("If this is 0, a model trained on one camera and tested on another camera of the SAME")
print("subject+state would be trivially 'correct' just by re-using the shared label.")
print()

# Worked example: a correct subject-level split
from numpy.random import default_rng
rng = default_rng(42)
subjects = df.patient_id.unique()
rng.shuffle(subjects)
n_test = int(len(subjects) * 0.2)
test_subjects = set(subjects[:n_test])
train_subjects = set(subjects[n_test:])

overlap = train_subjects & test_subjects
print(f"Worked split: {len(train_subjects)} train subjects, {len(test_subjects)} test subjects")
print(f"CHECK: subject overlap between train/test: {len(overlap)} (must be 0)")


CHECK: subject+state groups where targets differ across cameras: 0 (expected: 0)
If this is 0, a model trained on one camera and tested on another camera of the SAME
subject+state would be trivially 'correct' just by re-using the shared label.

Worked split: 480 train subjects, 120 test subjects
CHECK: subject overlap between train/test: 0 (must be 0)


## 12. Camera naming and file-size consistency
Confirms the camera-to-file-size pattern noted earlier (IriunWebcam ppg_sync files smaller
than FullHDwebcam/USBVideo), at full scale, and checks whether video files are real data or
still empty Git LFS placeholders.


In [15]:
ppg_sync_dir = DATA_DIR / "ppg_sync"
if ppg_sync_dir.exists():
    sizes = []
    for f in ppg_sync_dir.glob("*.txt"):
        # filenames look like 1020_FullHDwebcam_after.txt
        parts = f.stem.split("_")
        camera = parts[1] if len(parts) > 1 else "unknown"
        sizes.append({"camera": camera, "size_kb": f.stat().st_size / 1024})
    size_df = pd.DataFrame(sizes)
    print(size_df.groupby("camera")["size_kb"].describe())
else:
    print(f"{ppg_sync_dir} not found locally — skipping.")

print()

video_dir = DATA_DIR / "video"
if video_dir.exists():
    video_files = list(video_dir.glob("*.avi"))
    small = [f for f in video_files if f.stat().st_size < 1024]  # <1KB = still a placeholder
    print(f"Video files found: {len(video_files)}, still placeholders (<1KB): {len(small)}")
    if small[:5]:
        print("Example placeholder files (not yet pulled):", [f.name for f in small[:5]])
else:
    print(f"{video_dir} not found locally — skipping.")


               count       mean       std        min        25%        50%  \
camera                                                                       
FullHDwebcam  1200.0  68.631976  1.220156  43.138672  68.319092  68.652832   
IriunWebcam   1200.0  54.592867  1.155219  47.333984  54.325928  54.672852   
USBVideo      1200.0  68.677011  0.944737  65.267578  68.302734  68.655273   

                    75%        max  
camera                              
FullHDwebcam  68.995605  88.502930  
IriunWebcam   55.011963  70.640625  
USBVideo      69.006104  88.405273  

Video files found: 3600, still placeholders (<1KB): 3600
Example placeholder files (not yet pulled): ['1020_FullHDwebcam_after.avi', '1020_FullHDwebcam_before.avi', '1020_IriunWebcam_after.avi', '1020_IriunWebcam_before.avi', '1020_USBVideo_after.avi']


## 13. Summary — what to put in the report
Run this last to get a compact checklist of what passed, failed, or needs more data.


In [16]:
print("=" * 60)
print("SUMMARY CHECKLIST (fill in after running all cells above)")
print("=" * 60)
checklist = [
    "[ ] Row/subject counts match expected (3600 rows / 600 subjects)",
    "[ ] Glucose column confirmed absent",
    "[ ] Saturation narrow-range % calculated and noted",
    "[ ] Before/after change magnitudes confirmed at full scale",
    "[ ] Static-target pattern (label-leakage risk) confirmed at full scale",
    "[ ] ECG lead-length mismatch scanned across available files",
    "[ ] Flat-line PPG/ECG scan run on available files",
    "[ ] Low-saturation subjects' heart rates checked as plausible",
    "[ ] Subject-level split leakage risk proven with a self-test",
    "[ ] Camera/file-size pattern confirmed; video placeholder status checked",
]
for item in checklist:
    print(item)


SUMMARY CHECKLIST (fill in after running all cells above)
[ ] Row/subject counts match expected (3600 rows / 600 subjects)
[ ] Glucose column confirmed absent
[ ] Saturation narrow-range % calculated and noted
[ ] Before/after change magnitudes confirmed at full scale
[ ] Static-target pattern (label-leakage risk) confirmed at full scale
[ ] ECG lead-length mismatch scanned across available files
[ ] Flat-line PPG/ECG scan run on available files
[ ] Low-saturation subjects' heart rates checked as plausible
[ ] Subject-level split leakage risk proven with a self-test
[ ] Camera/file-size pattern confirmed; video placeholder status checked
